# Экспорт данных с дашбордов и сборка сводной таблицы

Ноутбук делает следующее:

1. Читает справочник дашбордов (Excel) с колонками `Дэш`, `Экран`, `Ссылка`.
2. Последовательно открывает каждую ссылку в Chrome (через Selenium).
3. Ждёт 20 секунд, чтобы виджеты прогрузились.
4. Эмулирует **Ctrl+A / Ctrl+C** реальными нажатиями клавиш и сохраняет скопированный текст в `txt/<сегодняшняя_дата>/<Дэш>_<Экран>.txt`.
5. Проходит по всем строкам справочника.
6. Разбирает txt-файлы на отдельные виджеты (показатели) и собирает сводный Excel:
   - Лист 1 «Показатели» — Показатель, Дэш, Экран, Ссылка, Актуально, Гранулярность, Есть прогноз, Есть недельный срез.
   - Лист 2 «Значения по периодам» — Показатель, Дэш, Экран, Период, Факт, Выполнение плана.

**Установка зависимостей:**
```
pip install pandas openpyxl selenium pyperclip
```
Linux: `sudo apt install xclip` (иначе буфер обмена не работает).

## Как устроен парсер (по двум реальным примерам)

Скопированный текст на самом деле построчный (по одному значению на строку). Видел уже 2 разных виджета:

**Виджет с единицей измерения и месяцами (Чистая прибыль):**
```
Чистая прибыль / Млн руб / Прогноз / 5.0 / План / 67) / Дельта нед. / -0.1
Факт/прогноз / План / Вып.
Авг2025 Сен Окт Ноя Дек Янв2026 Фев ... Дек    <- 17 месяцев (год - только при смене)
3,4 3,5 ... 5.4                                  <- 17 значений факта
98% 97% ... 89%                                   <- 8 значений выполнения плана (последние 8 месяцев)
```

**Виджет без единицы измерения, квартальный (CSI):**
```
CSI / Прогноз / План / Вып-е / 114% / нед. / Факт
3Q2025 4Q 1Q2026 2Q 3Q     <- 5 кварталов (год - только при смене)
10.0 11.6 14.6 15.7 16.5     <- 5 значений факта
45% 65% 86% 77% 88%           <- 5 значений выполнения плана
```

Вывод: заголовок виджета — это НЕ обязательно строка перед единицей измерения (у CSI единицы нет вообще), а строка перед **первым служебным словом** (единица измерения ИЛИ Прогноз/План/Факт/Вып/Дельта). Причём такое "защёлкивание" заголовка происходит только один раз на виджет (после первого попадания на служебное слово парсер ищет следующий заголовок только после того, как закончатся числовые данные текущего виджета) — иначе слова План/Вып, которые встречаются по несколько раз, постоянно сбивали бы заголовок.

⚠️ **Ограничения:**
- Если название показателя само начинается с одного из служебных слов (План.../Прогноз.../Факт...) — заголовок определится неверно. Пока таких примеров не было.
- «Актуально» вычисляется как **последний период (месяц/квартал) ≤ сегодняшней дате** — текст не позволяет отличить факт от прогноза внутри объединённой линии "Факт/прогноз".

In [ ]:
import os
import re
import time
from pathlib import Path
from datetime import date

import pandas as pd

# ================== НАСТРОЙКИ ==================
REFERENCE_XLSX = "справочник_дашей.xlsx"
SHEET_NAME = 0
WAIT_SECONDS = 20

OUTPUT_TXT_ROOT = Path("txt")
TODAY_FOLDER = OUTPUT_TXT_ROOT / date.today().isoformat()
RESULT_XLSX = f"Сводная_таблица_{date.today().isoformat()}.xlsx"

CHROME_USER_DATA_DIR = None  # путь к профилю Chrome, если нужна авторизация
CHROME_PROFILE = "Default"

TODAY_FOLDER.mkdir(parents=True, exist_ok=True)
print("Файлы будут сохранены в:", TODAY_FOLDER.resolve())

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.action_chains import ActionChains


def create_driver():
    options = webdriver.ChromeOptions()
    if CHROME_USER_DATA_DIR:
        options.add_argument(f"--user-data-dir={CHROME_USER_DATA_DIR}")
        options.add_argument(f"--profile-directory={CHROME_PROFILE}")
    options.add_argument("--start-maximized")
    driver = webdriver.Chrome(options=options)
    return driver


driver = create_driver()

In [ ]:
import pyperclip


def copy_page_content(driver, pause=0.4):
    '''Эмулирует выделение всей страницы (Ctrl+A) и копирование (Ctrl+C).'''
    pyperclip.copy("")
    body = driver.find_element(By.TAG_NAME, "body")
    body.click()
    actions = ActionChains(driver)
    actions.key_down(Keys.CONTROL).send_keys("a").key_up(Keys.CONTROL).perform()
    time.sleep(pause)
    actions.key_down(Keys.CONTROL).send_keys("c").key_up(Keys.CONTROL).perform()
    time.sleep(pause)

    text = pyperclip.paste()
    if not text.strip():
        time.sleep(1)
        text = pyperclip.paste()
    return text


def safe_filename(name: str) -> str:
    name = str(name).strip()
    name = re.sub(r'[\\/*?:"<>|]', "_", name)
    return name

## Шаг 1-6. Проход по всем ссылкам и сохранение txt-файлов

In [ ]:
ref_df = pd.read_excel(REFERENCE_XLSX, sheet_name=SHEET_NAME)
ref_df = ref_df.rename(columns=lambda c: str(c).strip())

required_cols = {"Дэш", "Экран", "Ссылка"}
missing = required_cols - set(ref_df.columns)
if missing:
    raise ValueError(f"В справочнике не хватает колонок: {missing}")

errors = []

for i, row in ref_df.iterrows():
    dash = str(row["Дэш"]).strip()
    screen = str(row["Экран"]).strip()
    url = str(row["Ссылка"]).strip()

    fname = f"{safe_filename(dash)}_{safe_filename(screen)}.txt"
    fpath = TODAY_FOLDER / fname

    print(f"[{i + 1}/{len(ref_df)}] {dash} / {screen} -> {url}")

    try:
        driver.get(url)
        time.sleep(WAIT_SECONDS)
        content = copy_page_content(driver)

        if not content.strip():
            print(f"  ⚠️ Пусто! Проверьте вручную: {url}")

        fpath.write_text(content, encoding="utf-8")
    except Exception as e:
        print(f"  ❌ Ошибка: {e}")
        errors.append((dash, screen, url, str(e)))

print("Готово. Файлы сохранены в:", TODAY_FOLDER)
if errors:
    print("Ошибки:")
    for e in errors:
        print(" ", e)

In [ ]:
driver.quit()

## Шаг 7. Парсинг txt-файлов

Настройки ниже (единицы измерения, служебные слова) можно и нужно дополнять под свои дашборды.

In [ ]:
MONTHS_RU = {
    "янв": 1, "января": 1, "январь": 1,
    "фев": 2, "февраля": 2, "февраль": 2,
    "мар": 3, "марта": 3, "март": 3,
    "апр": 4, "апреля": 4, "апрель": 4,
    "май": 5, "мая": 5,
    "июн": 6, "июня": 6, "июнь": 6,
    "июл": 7, "июля": 7, "июль": 7,
    "авг": 8, "августа": 8, "август": 8,
    "сен": 9, "сентября": 9, "сентябрь": 9,
    "окт": 10, "октября": 10, "октябрь": 10,
    "ноя": 11, "ноября": 11, "ноябрь": 11,
    "дек": 12, "декабря": 12, "декабрь": 12,
}

# месяц с явным годом: "авг2025", "янв.24", "январь 2024", "01.2024", "2024-01"
MONTH_WITH_YEAR_RE = re.compile(
    r"^(?:"
    r"(?P<name>[а-яё]+)\.?\s*['`]?(?P<y1>\d{2,4})"
    r"|(?P<mm>\d{1,2})[./-](?P<y2>\d{2,4})"
    r"|(?P<y3>\d{4})[./-](?P<mm2>\d{1,2})"
    r")$",
    re.IGNORECASE,
)
BARE_MONTH_RE = re.compile(r"^[а-яё]+$", re.IGNORECASE)

# квартал: "3Q2025", "4Q", "1кв2026", "2 кв. 25"
QUARTER_RE = re.compile(
    r"^(?P<num>[1-4])\s*(?:q|кв)\.?\s*(?P<year>\d{2,4})?$",
    re.IGNORECASE,
)

NUMBER_RE = re.compile(r"^-?\d[\d\s.,]*%?$")

FORECAST_KEYWORDS = ("прогноз",)
WEEKLY_KEYWORDS = ("нед",)

# единицы измерения
UNIT_PATTERNS = [
    re.compile(r"^(млн|тыс|млрд)\.?\s*руб\.?$", re.IGNORECASE),
    re.compile(r"^руб\.?$", re.IGNORECASE),
    re.compile(r"^%$"),
    re.compile(r"^(млн|тыс|млрд)?\.?\s*шт\.?$", re.IGNORECASE),
    re.compile(r"^(млн|тыс|млрд)?\.?\s*ед\.?$", re.IGNORECASE),
    # добавьте сюда свои варианты единиц измерения, если парсер их не находит
]

# служебные слова-подписи (легенда/KPI-плашки), которые НЕ являются заголовком показателя,
# а сигнализируют, что заголовок только что закончился (см. is_label_or_unit_line)
LABEL_STEMS = ("прогноз", "план", "факт", "выполнение", "вып", "дельта")


def is_unit_line(token: str) -> bool:
    t = token.strip()
    return any(p.match(t) for p in UNIT_PATTERNS)


def _normalize_label(token: str) -> str:
    t = token.strip().lower()
    t = re.sub(r"[.\-):]+$", "", t)
    t = re.sub(r"^[.\-(:]+", "", t)
    return t


def is_label_or_unit_line(token: str) -> bool:
    if is_unit_line(token):
        return True
    t = _normalize_label(token)
    return any(t == stem or t.startswith(stem) for stem in LABEL_STEMS)


def _month_name_to_num(name: str):
    name = name.lower()
    for key, num in MONTHS_RU.items():
        if name.startswith(key):
            return num
    return None


def parse_period_token(token: str, current_year):
    '''
    Пытается распознать токен как месяц или квартал (с явным годом или без - тогда
    используется "протянутый" current_year). Возвращает ((тип, год, номер), новый_год)
    либо (None, current_year). тип = "M" (месяц) или "Q" (квартал).
    '''
    t = token.strip().lower()

    m = MONTH_WITH_YEAR_RE.match(t)
    if m:
        if m.group("name"):
            month = _month_name_to_num(m.group("name"))
            year = m.group("y1")
        elif m.group("mm"):
            month = int(m.group("mm"))
            year = m.group("y2")
        else:
            month = int(m.group("mm2"))
            year = m.group("y3")
        if month is not None and 1 <= month <= 12:
            year = int(year)
            if year < 100:
                year += 2000
            return ("M", year, month), year

    q = QUARTER_RE.match(t)
    if q:
        num = int(q.group("num"))
        year = q.group("year")
        if year is not None:
            year = int(year)
            if year < 100:
                year += 2000
            return ("Q", year, num), year
        elif current_year is not None:
            return ("Q", current_year, num), current_year

    if BARE_MONTH_RE.match(t) and current_year is not None:
        month = _month_name_to_num(t)
        if month is not None:
            return ("M", current_year, month), current_year

    return None, current_year


def split_line(line: str):
    if "\t" in line:
        parts = line.split("\t")
    else:
        parts = re.split(r"\s{2,}", line)
    return [p.strip() for p in parts if p.strip() != ""]


def to_number(s: str):
    s = s.replace("\xa0", "").replace(" ", "").replace("%", "")
    s = s.replace(",", ".")
    try:
        return float(s)
    except ValueError:
        return None


def flatten_tokens(text: str):
    tokens = []
    for raw_line in text.splitlines():
        line = raw_line.strip()
        if not line:
            continue
        tokens.extend(split_line(line))
    return tokens


def format_period(p):
    ptype, year, num = p
    if ptype == "M":
        return f"{num:02d}.{year}"
    return f"{num}кв.{year}"


def period_sort_key(p):
    return (p[1], p[2])

In [ ]:
def parse_widgets(text: str):
    '''
    Разбирает текст, скопированный с экрана, на отдельные виджеты (показатели).
    Возвращает список словарей:
      {"title", "forecast", "weekly", "periods": [(тип, год, номер), ...], "fact": [...], "plan": [...]}
    '''
    tokens = flatten_tokens(text)
    n = len(tokens)
    widgets = []

    pending_meta = []
    current = None
    current_year = None
    awaiting_title = True  # ищем ли сейчас заголовок нового виджета

    def new_widget(title):
        return {"title": title, "forecast": False, "weekly": False,
                "periods": [], "fact": [], "plan": []}

    def scan_keywords(widget, token):
        low = token.lower()
        if any(k in low for k in FORECAST_KEYWORDS):
            widget["forecast"] = True
        if any(k in low for k in WEEKLY_KEYWORDS):
            widget["weekly"] = True

    i = 0
    while i < n:
        tok = tokens[i]

        # 1) первое служебное слово после предыдущих данных -> заголовок только что закончился
        if awaiting_title and is_label_or_unit_line(tok) and pending_meta:
            if current and current["periods"]:
                widgets.append(current)
            current = new_widget(pending_meta[-1])
            current_year = None
            scan_keywords(current, tok)
            pending_meta = []
            awaiting_title = False
            i += 1
            continue

        # 2) период (месяц или квартал, с годом или без)
        parsed, y2 = (None, current_year)
        if current is not None:
            parsed, y2 = parse_period_token(tok, current_year)

        if parsed is not None:
            periods_buf = [parsed]
            current_year = y2
            i += 1
            while i < n:
                p2, y3 = parse_period_token(tokens[i], current_year)
                if p2 is None or p2[0] != parsed[0]:  # не смешиваем месяцы и кварталы
                    break
                periods_buf.append(p2)
                current_year = y3
                i += 1
            current["periods"] = periods_buf

            def consume_numbers(limit):
                nonlocal i
                vals = []
                while i < n and len(vals) < limit and NUMBER_RE.match(tokens[i]):
                    vals.append(to_number(tokens[i]))
                    i += 1
                return vals

            current["fact"] = consume_numbers(len(periods_buf))
            current["plan"] = consume_numbers(len(periods_buf))
            awaiting_title = True  # данные виджета закончились - дальше ищем новый заголовок
            continue

        # 3) ключевые слова (прогноз / недельный срез) сканируем на лету
        if current is not None:
            scan_keywords(current, tok)

        # 4) обычный текст - кандидат в заголовок следующего виджета
        pending_meta.append(tok)
        i += 1

    if current and current["periods"]:
        widgets.append(current)

    return widgets

### Проверка парсера на реальных примерах

Ниже - самопроверка на тексте из ваших файлов-примеров (месячный показатель «Чистая прибыль» + квартальный «CSI»).

In [ ]:
_sample_text = '''Обзор
Монитор бизнеса
Цели
ДИб
БИБ
ХУБ
Чистая прибыль
Млн руб
Прогноз
5.0
План
67)
Дельта нед.
-0.1
Факт/прогноз
План
Вып.
Авг2025
Сен
Окт
Ноя
Дек
Янв2026
Фев
Мар
Апр
Май
Июн
Июл
Авг
Сен
Окт
Ноя
Дек
3,4
3,5
3,6
3.7
3.8
3.9
4.2
4.3
4.4
4.5
4.5
4.6
5.7
5.3
4.7
5.3
5.4
98%
97%
95%
44%
85%
86%
88%
89%
CSI
Прогноз
План
Вып-е
114%
нед.
Факт
3Q2025
4Q
1Q2026
2Q
3Q
10.0
11.6
14.6
15.7
16.5
45%
65%
86%
77%
88%
'''

_test_widgets = parse_widgets(_sample_text)
for w in _test_widgets:
    print("Показатель:", w["title"])
    print("  Прогноз:", w["forecast"], "| Недельный срез:", w["weekly"])
    print("  Гранулярность:", "Месяц" if w["periods"][0][0] == "M" else "Квартал")
    print("  Периодов:", len(w["periods"]), "| Факт:", len(w["fact"]), "| План:", len(w["plan"]))
    print("  Первый период:", format_period(w["periods"][0]), "Последний:", format_period(w["periods"][-1]))
    print()

## Шаг 8. Сборка итогового Excel по всем txt-файлам

In [ ]:
def compute_actual_period(periods, fact, today=None):
    '''
    'Актуально' = последний период (месяц/квартал) <= сегодняшней дате, для которого есть факт.
    Прокси-логика: текстовая копия не позволяет однозначно отличить факт от прогноза
    внутри объединённой серии "Факт/прогноз".
    '''
    today = today or date.today()
    today_q = (today.month - 1) // 3 + 1
    dated = [p for p, val in zip(periods, fact) if val is not None]
    if not dated:
        return None

    def is_past_or_present(p):
        ptype, year, num = p
        if ptype == "M":
            return (year, num) <= (today.year, today.month)
        return (year, num) <= (today.year, today_q)

    past = [p for p in dated if is_past_or_present(p)]
    pool = past if past else dated
    return max(pool, key=period_sort_key)


summary_rows = []
period_rows = []

for _, row in ref_df.iterrows():
    dash = str(row["Дэш"]).strip()
    screen = str(row["Экран"]).strip()
    url = str(row["Ссылка"]).strip()
    fname = f"{safe_filename(dash)}_{safe_filename(screen)}.txt"
    fpath = TODAY_FOLDER / fname

    if not fpath.exists():
        print(f"Пропускаю (нет файла): {fpath}")
        continue

    text = fpath.read_text(encoding="utf-8")
    widgets = parse_widgets(text)

    if not widgets:
        print(f"⚠️ Ни одного виджета не распознано в {fpath.name} - проверьте формат файла")

    for w in widgets:
        periods = w["periods"]
        fact = w["fact"]
        plan = w["plan"]

        if not periods or not fact:
            continue

        actual_period = compute_actual_period(periods, fact)
        granularity = "Месяц" if periods[0][0] == "M" else "Квартал"

        summary_rows.append({
            "Показатель": w["title"],
            "Дэш": dash,
            "Экран": screen,
            "Ссылка": url,
            "Актуально": format_period(actual_period) if actual_period else "",
            "Гранулярность": granularity,
            "Есть прогноз": "Да" if w["forecast"] else "Нет",
            "Есть недельный срез": "Да" if w["weekly"] else "Нет",
        })

        # план короче факта -> выравниваем по максимальным (последним) датам
        plan_periods = periods[len(periods) - len(plan):] if plan else []
        plan_dict = dict(zip(plan_periods, plan))

        for idx, p in enumerate(periods):
            fact_val = fact[idx] if idx < len(fact) else None
            plan_val = plan_dict.get(p)
            period_rows.append({
                "Показатель": w["title"],
                "Дэш": dash,
                "Экран": screen,
                "Гранулярность": granularity,
                "Период": format_period(p),
                "Год": p[1],
                "СортировкаПериода": p[2],
                "Факт": fact_val,
                "Выполнение плана": plan_val,
            })

summary_df = pd.DataFrame(summary_rows)
period_df = pd.DataFrame(period_rows)

with pd.ExcelWriter(RESULT_XLSX, engine="openpyxl") as writer:
    summary_df.to_excel(writer, sheet_name="Показатели", index=False)
    period_df.to_excel(writer, sheet_name="Значения по периодам", index=False)

print("Сохранено:", RESULT_XLSX)
summary_df.head()